In [1]:
BRONZE_BASE = (
    "abfss://itransition_de_project@onelake.dfs.fabric.microsoft.com"
    "/bronze.Lakehouse"
)

bronze_gdp_path = f"{BRONZE_BASE}/Tables/dbo/gdp_raw"
bronze_fx_path  = f"{BRONZE_BASE}/Tables/dbo/fx_raw"

silver_gdp_table = "gdp"
silver_fx_table  = "fx_daily"
silver_gdp_fx    = "gdp_fx"
write_mode       = "overwrite"

FX_MIN = 0.50
FX_MAX = 2.00

StatementMeta(, 839dffea-ee73-4b80-9598-856d539f6a70, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType, DateType

spark = SparkSession.builder.getOrCreate()

StatementMeta(, 839dffea-ee73-4b80-9598-856d539f6a70, 5, Finished, Available, Finished, False)

In [4]:
raw_gdp = spark.read.format("delta").load(bronze_gdp_path)

print(f"Bronze GDP rows : {raw_gdp.count():,}")
raw_gdp.printSchema()

gdp_df = (
    raw_gdp
    .withColumn("year",         F.col("date").cast(IntegerType()))
    .withColumn("gdp_usd",      F.col("value").cast(DoubleType()))
    .withColumn("country_name", F.col("country.value").cast(StringType()))
    .withColumn("country_iso3", F.col("countryiso3code").cast(StringType()))
    .select("country_iso3", "country_name", "year", "gdp_usd", "_ingested_at")
    .filter(F.col("year").isNotNull())
    .filter(F.col("gdp_usd").isNotNull())
    .filter(F.col("gdp_usd") > 0)
    .dropDuplicates(["country_iso3", "year"])
    .orderBy("country_iso3", "year")
)

gdp_count = gdp_df.count()
print(f"\nClean GDP rows  : {gdp_count:,}")

(
    gdp_df.write
    .format("delta")
    .mode(write_mode)
    .option("overwriteSchema", "true")
    .saveAsTable(silver_gdp_table)
)
print(f"[OK] silver.{silver_gdp_table} written")

StatementMeta(, 839dffea-ee73-4b80-9598-856d539f6a70, 7, Finished, Available, Finished, False)

Bronze GDP rows : 450
root
 |-- _country_iso: string (nullable = true)
 |-- country: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- value: string (nullable = true)
 |-- countryiso3code: string (nullable = true)
 |-- date: string (nullable = true)
 |-- decimal: long (nullable = true)
 |-- indicator: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- value: string (nullable = true)
 |-- obs_status: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: double (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)


Clean GDP rows  : 90
[OK] silver.gdp written


In [5]:
raw_fx = spark.read.format("delta").load(bronze_fx_path)

print(f"\nBronze FX rows  : {raw_fx.count():,}")

date_col = "rate_date_str"   if "rate_date_str"   in raw_fx.columns else "TIME_PERIOD"
rate_col = "usd_eur_rate_raw" if "usd_eur_rate_raw" in raw_fx.columns else "OBS_VALUE"

fx_df = (
    raw_fx
    .withColumn("rate_date",    F.to_date(F.col(date_col), "yyyy-MM-dd").cast(DateType()))
    .withColumn("usd_eur_rate", F.col(rate_col).cast(DoubleType()))
    .select("rate_date", "usd_eur_rate", "_ingested_at")
    .filter(F.col("rate_date").isNotNull())
    .filter(F.col("usd_eur_rate").isNotNull())
    .filter(F.col("usd_eur_rate").between(FX_MIN, FX_MAX))
    .dropDuplicates(["rate_date"])
    .orderBy("rate_date")
)

fx_count = fx_df.count()
print(f"Clean FX rows   : {fx_count:,}")

(
    fx_df.write
    .format("delta")
    .mode(write_mode)
    .option("overwriteSchema", "true")
    .saveAsTable(silver_fx_table)
)
print(f"[OK] silver.{silver_fx_table} written")

StatementMeta(, 839dffea-ee73-4b80-9598-856d539f6a70, 8, Finished, Available, Finished, False)


Bronze FX rows  : 3,851
Clean FX rows   : 3,842
[OK] silver.fx_daily written


In [6]:
fx_yearly_df = (
    fx_df
    .withColumn("year", F.year("rate_date").cast(IntegerType()))
    .groupBy("year")
    .agg(
        F.round(F.avg("usd_eur_rate"), 6).alias("avg_usd_eur_rate"),
        F.round(F.min("usd_eur_rate"), 6).alias("min_usd_eur_rate"),
        F.round(F.max("usd_eur_rate"), 6).alias("max_usd_eur_rate"),
        F.count("*").alias("trading_days"),
    )
    .orderBy("year")
)

print("\nYearly FX averages:")
fx_yearly_df.show()


StatementMeta(, 839dffea-ee73-4b80-9598-856d539f6a70, 9, Finished, Available, Finished, False)


Yearly FX averages:
+----+----------------+----------------+----------------+------------+
|year|avg_usd_eur_rate|min_usd_eur_rate|max_usd_eur_rate|trading_days|
+----+----------------+----------------+----------------+------------+
|2010|        1.325717|          1.1942|          1.4563|         258|
|2011|        1.391955|          1.2889|          1.4882|         257|
|2012|        1.284789|          1.2089|          1.3454|         256|
|2013|        1.328118|          1.2768|          1.3814|         255|
|2014|        1.328501|          1.2141|          1.3953|         255|
|2015|        1.109513|          1.0552|          1.2043|         256|
|2016|        1.106903|          1.0364|          1.1569|         257|
|2017|        1.129681|          1.0385|           1.206|         255|
|2018|        1.180955|          1.1261|          1.2493|         255|
|2019|        1.119475|          1.0889|          1.1535|         255|
|2020|        1.142196|          1.0707|          1.2281

In [7]:
gdp_fx_df = (
    gdp_df
    .drop("_ingested_at")
    .join(fx_yearly_df, on="year", how="left")
    # GDP in EUR = GDP in USD ÷ USD_per_EUR rate
    # ECB quote: OBS_VALUE = USD per 1 EUR  →  EUR = USD / rate
    .withColumn(
        "gdp_eur",
        F.when(
            F.col("avg_usd_eur_rate").isNotNull() & (F.col("avg_usd_eur_rate") > 0),
            F.round(F.col("gdp_usd") / F.col("avg_usd_eur_rate"), 2)
        )
    )
    .orderBy("country_iso3", "year")
)

gdp_fx_count = gdp_fx_df.count()
print(f"\ngdp_fx rows : {gdp_fx_count:,}")

(
    gdp_fx_df.write
    .format("delta")
    .mode(write_mode)
    .option("overwriteSchema", "true")
    .saveAsTable(silver_gdp_fx)
)
print(f"[OK] silver.{silver_gdp_fx} written")

StatementMeta(, 839dffea-ee73-4b80-9598-856d539f6a70, 11, Finished, Available, Finished, False)


gdp_fx rows : 90
[OK] silver.gdp_fx written


In [8]:
print("\n--- GDP Summary ---")
spark.sql(f"""
    SELECT country_iso3, COUNT(*) AS years,
           ROUND(MIN(gdp_usd)/1e12, 2) AS min_gdp_tn,
           ROUND(MAX(gdp_usd)/1e12, 2) AS max_gdp_tn
    FROM {silver_gdp_table}
    GROUP BY 1
    ORDER BY 1
""").show()

print("--- FX Summary ---")
spark.sql(f"""
    SELECT
        MIN(rate_date) AS earliest,
        MAX(rate_date) AS latest,
        COUNT(*)       AS days,
        ROUND(AVG(usd_eur_rate), 4) AS avg_rate
    FROM {silver_fx_table}
""").show()

print("--- GDP+FX spot check ---")
spark.sql(f"""
    SELECT country_iso3, year, ROUND(gdp_usd/1e12,2) AS gdp_tn_usd,
           avg_usd_eur_rate, ROUND(gdp_eur/1e12,2) AS gdp_tn_eur
    FROM {silver_gdp_fx}
    WHERE country_iso3 = 'USA'
    ORDER BY year DESC
    LIMIT 5
""").show()

StatementMeta(, 839dffea-ee73-4b80-9598-856d539f6a70, 12, Finished, Available, Finished, False)


--- GDP Summary ---
+------------+-----+----------+----------+
|country_iso3|years|min_gdp_tn|max_gdp_tn|
+------------+-----+----------+----------+
|         CHN|   15|      6.19|     18.74|
|         DEU|   15|      3.43|      4.69|
|         FRA|   15|      2.44|      3.16|
|         GBR|   15|       2.5|      3.69|
|         JPN|   15|      4.03|      6.27|
|         USA|   15|     15.05|     28.75|
+------------+-----+----------+----------+

--- FX Summary ---
+----------+----------+----+--------+
|  earliest|    latest|days|avg_rate|
+----------+----------+----+--------+
|2010-01-04|2024-12-31|3842|  1.1899|
+----------+----------+----+--------+

--- GDP+FX spot check ---
+------------+----+----------+----------------+----------+
|country_iso3|year|gdp_tn_usd|avg_usd_eur_rate|gdp_tn_eur|
+------------+----+----------+----------------+----------+
|         USA|2024|     28.75|         1.08238|     26.56|
|         USA|2023|     27.29|        1.081269|     25.24|
|         USA|202